# GroceryCo Audit - 03 - Skewness Analysis & SPC Control Charts

**Task 2.** Use NumPy and pandas to compute distribution statistics (skewness, kurtosis) for operational metrics, then apply Statistical Process Control (SPC) to identify departments whose reorder rates are 'out of control' relative to the platform mean.

**SPC framework:** for each department, we treat each day-of-week as a subgroup observation, compute the mean reorder rate, and place ±3-sigma control limits around the platform-wide mean. Departments breaching the limits or showing Western Electric pattern violations are flagged as out-of-control.

**Why this is the logistics-hub analogue:** the brief asked for control charts on logistics hubs - we use departments as the GroceryCo equivalent of operational units to monitor.

## Setup - auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "grocery_audit":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPECTED = ["aisles.csv", "departments.csv", "products.csv", "orders.csv", "order_products__train.csv"]
csv_paths = {}
for name in EXPECTED:
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[name] = folder / name
            break
missing = [f for f in EXPECTED if f not in csv_paths]
assert not missing, f"Missing CSVs: {missing}. Place them in {DATASET_DIR} or {DATA_DIR}"

print(f"Project root : {PROJECT_ROOT}")
print(f"Found {len(csv_paths)} CSVs")


## Step 1 - Load data and compute distribution statistics

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

conn = sqlite3.connect(OUTPUTS_DIR / "groceryco.db")

# Items per order
items_per_order = pd.read_sql_query('''
    SELECT order_id, COUNT(*) AS n_items
    FROM order_items
    GROUP BY order_id
''', conn)
print(f"Computed items per order for {len(items_per_order):,} orders")

# Days since prior order
dspo = pd.read_sql_query('''
    SELECT days_since_prior_order
    FROM orders
    WHERE days_since_prior_order IS NOT NULL
''', conn)
print(f"Loaded {len(dspo):,} days_since_prior_order observations")


## Step 2 - Skewness and kurtosis on operational metrics

In [ ]:
def describe_distribution(name, series):
    s = series.dropna()
    print(f"\n=== {name} (n={len(s):,}) ===")
    print(f"  mean      = {s.mean():>10.3f}")
    print(f"  median    = {s.median():>10.3f}")
    print(f"  std       = {s.std():>10.3f}")
    print(f"  min/max   = {s.min():.0f} / {s.max():.0f}")
    print(f"  skewness  = {s.skew():>10.3f}  (0 = symmetric, >0 = right tail, <0 = left tail)")
    print(f"  kurtosis  = {s.kurtosis():>10.3f}  (0 = normal, >0 = heavy tails)")
    # Jarque-Bera normality test
    jb_stat, jb_p = stats.jarque_bera(s.sample(min(5000, len(s)), random_state=42))
    normal = "NOT normal" if jb_p < 0.05 else "consistent with normal"
    print(f"  Jarque-Bera p = {jb_p:.4e} -> {normal}")

describe_distribution("Items per order", items_per_order["n_items"])
describe_distribution("Days since prior order", dspo["days_since_prior_order"])


## Step 3 - Visualise the distributions (with skew annotations)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Items per order
ax = axes[0]
ax.hist(items_per_order["n_items"], bins=50, color="#1F77B4",
        alpha=0.75, edgecolor="white")
mean_v = items_per_order["n_items"].mean()
median_v = items_per_order["n_items"].median()
ax.axvline(mean_v, color="red", linewidth=2, label=f"mean={mean_v:.2f}")
ax.axvline(median_v, color="orange", linewidth=2, linestyle="--", label=f"median={median_v:.0f}")
ax.set_title(f"Items per order  (skew={items_per_order['n_items'].skew():.2f})",
             fontweight="bold")
ax.set_xlabel("Items per order"); ax.set_ylabel("Frequency")
ax.legend()

# Days since prior order
ax = axes[1]
ax.hist(dspo["days_since_prior_order"], bins=31, color="#2E7D7D",
        alpha=0.75, edgecolor="white")
mean_v = dspo["days_since_prior_order"].mean()
median_v = dspo["days_since_prior_order"].median()
ax.axvline(mean_v, color="red", linewidth=2, label=f"mean={mean_v:.2f}")
ax.axvline(median_v, color="orange", linewidth=2, linestyle="--", label=f"median={median_v:.0f}")
ax.axvline(30, color="grey", linewidth=1.5, linestyle=":", label="30-day cap")
ax.set_title(f"Days since prior order  (skew={dspo['days_since_prior_order'].skew():.2f})",
             fontweight="bold")
ax.set_xlabel("Days"); ax.set_ylabel("Frequency")
ax.legend()

plt.tight_layout()
plt.savefig(FIGURES_DIR / "distributions_skewness.png", dpi=160, bbox_inches="tight")
plt.show()


## Step 4 - Build the SPC observation matrix

For each department × each day-of-week (the subgroup), compute the mean reorder rate. This gives us 21 × 7 = 147 observations to monitor. The platform-wide mean and std across these subgroups become our control chart centerline and limits.

In [ ]:
spc_data = pd.read_sql_query('''
    SELECT
        d.department,
        o.order_dow,
        AVG(oi.reordered) AS reorder_rate,
        COUNT(*)          AS n_items
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p     ON oi.product_id = p.product_id
    JOIN departments d  ON p.department_id = d.department_id
    GROUP BY d.department, o.order_dow
    ORDER BY d.department, o.order_dow
''', conn)
print(f"SPC observation matrix: {len(spc_data)} rows ({spc_data['department'].nunique()} departments x 7 days)")
print(spc_data.head(14).to_string(index=False))


## Step 5 - Compute control limits

**SPC framing.** We treat each department as a peer observation in a 21-department operational system. The audit question is: *which departments deviate from the peer group's central behaviour by more than would be expected from random variation?*

**Two limits are computed:**
1. **Peer-spread limits** (±2σ on cross-department mean): a department whose mean reorder rate is >2σ from the platform mean is operating in a statistically distinct regime from its peers. (~5% expected by chance under normality.)
2. **Within-department stability** (each department's own ±3σ on its 7-day series): tells us whether each department is operating consistently across the week.

We use the peer-spread test for cross-department flagging (the headline audit finding) and Western Electric rules on the within-department series (in Step 7).

In [ ]:
# Per-department mean reorder rate (across the 7 day-of-week subgroups)
dept_stats = spc_data.groupby("department")["reorder_rate"].agg(
    ["mean", "std", "count"]
).round(4).reset_index()
dept_stats.columns = ["department", "x_bar", "subgroup_std", "n_subgroups"]

# Peer-spread approach: cross-department std as the noise floor
grand_mean = dept_stats["x_bar"].mean()
peer_std   = dept_stats["x_bar"].std()

# Use 2-sigma for OUT (95% peer band - typical for peer-benchmark audits)
# and 1-sigma for WARN (68% peer band)
ucl = grand_mean + 2 * peer_std
lcl = grand_mean - 2 * peer_std
uwl = grand_mean + 1 * peer_std
lwl = grand_mean - 1 * peer_std

print(f"Platform statistics (across {len(dept_stats)} departments):")
print(f"  Grand mean (CL)         = {grand_mean:.4f} ({grand_mean*100:.2f}%)")
print(f"  Peer std (cross-dept)   = {peer_std:.4f}")
print(f"  Upper Control Limit (+2sigma) = {ucl:.4f} ({ucl*100:.2f}%)")
print(f"  Upper Warning  (+1sigma)      = {uwl:.4f} ({uwl*100:.2f}%)")
print(f"  Lower Warning  (-1sigma)      = {lwl:.4f} ({lwl*100:.2f}%)")
print(f"  Lower Control Limit (-2sigma) = {lcl:.4f} ({lcl*100:.2f}%)")

# Flag out-of-control / warning departments
dept_stats["status"] = dept_stats["x_bar"].apply(
    lambda v: "OUT_HIGH" if v > ucl else
              "OUT_LOW"  if v < lcl else
              "WARN_HIGH" if v > uwl else
              "WARN_LOW"  if v < lwl else
              "IN_CONTROL"
)
n_out  = (dept_stats["status"].str.startswith("OUT")).sum()
n_warn = (dept_stats["status"].str.startswith("WARN")).sum()
print(f"\n{n_out} OUT-of-peer-band, {n_warn} in WARNING band, "
      f"{len(dept_stats) - n_out - n_warn} in central peer band.")
print(f"\nDepartment status:")
print(dept_stats.sort_values("x_bar", ascending=False).to_string(index=False))

dept_stats.to_csv(OUTPUTS_DIR / "spc_department_stats.csv", index=False)


## Step 6 - Plot the X-bar control chart

In [ ]:
dept_sorted = dept_stats.sort_values("x_bar", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 7))
x = range(len(dept_sorted))
colors = []
for s in dept_sorted["status"]:
    if s.startswith("OUT"): colors.append("#C00000")
    elif s.startswith("WARN"): colors.append("#E07A1F")
    else: colors.append("#2E7D32")

ax.scatter(x, dept_sorted["x_bar"], s=120, c=colors, edgecolor="black",
           linewidth=1.2, zorder=5)

# Limit lines (using ucl/lcl/uwl/lwl computed in Step 5 with sigma_xbar)
ax.axhline(grand_mean, color="#1F3864", linewidth=2, label=f"CL = {grand_mean:.3f}")
ax.axhline(ucl, color="#C00000", linewidth=1.5, linestyle="--", label=f"UCL = {ucl:.3f}")
ax.axhline(lcl, color="#C00000", linewidth=1.5, linestyle="--", label=f"LCL = {lcl:.3f}")
ax.axhline(uwl, color="#E07A1F", linewidth=1.0, linestyle=":", label=f"UWL/LWL ({uwl:.3f}/{lwl:.3f})")
ax.axhline(lwl, color="#E07A1F", linewidth=1.0, linestyle=":")
ax.fill_between(x, lcl, ucl, color="#A9D08E", alpha=0.10, label="Control band")

# Labels for OUT departments
for i, row in dept_sorted.iterrows():
    if row["status"].startswith("OUT"):
        ax.annotate(row["department"], (i, row["x_bar"]),
                    xytext=(8, 8), textcoords="offset points",
                    fontsize=9, fontweight="bold", color="#C00000")

ax.set_xticks(x)
ax.set_xticklabels(dept_sorted["department"], rotation=70, ha="right", fontsize=9)
ax.set_ylabel("Mean reorder rate (across day-of-week subgroups)")
ax.set_title("X-bar Control Chart - Department Reorder Rates\n"
             "Out-of-control departments flagged in RED",
             fontweight="bold", pad=12)
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.set_ylim(min(dept_sorted["x_bar"].min(), lcl) - 0.05,
            max(dept_sorted["x_bar"].max(), ucl) + 0.05)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "control_chart_xbar.png", dpi=160, bbox_inches="tight")
plt.show()


## Step 7 - Western Electric pattern detection

Beyond simple ±3-sigma breaches, Western Electric rules detect more subtle instability:

- **Rule 1**: 1 point beyond ±3-sigma (a single extreme observation)
- **Rule 2**: 2 of 3 consecutive points beyond ±2-sigma on same side
- **Rule 3**: 4 of 5 consecutive points beyond ±1-sigma on same side
- **Rule 4**: 8 consecutive points on same side of centerline

We apply these to the per-department time series across the 7 days of week.

In [ ]:
# Build per-department time series across the 7 day-of-week subgroups
ts_pivot = spc_data.pivot(index="department", columns="order_dow", values="reorder_rate")

# Per-department sigma (across 7 days)
ts_pivot["x_bar"] = ts_pivot[[0,1,2,3,4,5,6]].mean(axis=1)
ts_pivot["sigma"] = ts_pivot[[0,1,2,3,4,5,6]].std(axis=1)

we_violations = []
for dept, row in ts_pivot.iterrows():
    cl = row["x_bar"]
    sd = row["sigma"]
    series = row[[0,1,2,3,4,5,6]].values

    # Rule 1: any point beyond +/- 3 sigma (within department)
    r1 = (np.abs(series - cl) > 3 * sd).any()
    # Rule 2: 2 of 3 consecutive beyond +/-2 sigma same side
    r2 = False
    for i in range(len(series) - 2):
        window = series[i:i+3]
        n_above_2sig = ((window - cl) > 2 * sd).sum()
        n_below_2sig = ((cl - window) > 2 * sd).sum()
        if n_above_2sig >= 2 or n_below_2sig >= 2:
            r2 = True; break
    # Rule 3: 4 of 5 consecutive beyond +/-1 sigma same side
    r3 = False
    for i in range(len(series) - 4):
        window = series[i:i+5]
        n_above_1sig = ((window - cl) > sd).sum()
        n_below_1sig = ((cl - window) > sd).sum()
        if n_above_1sig >= 4 or n_below_1sig >= 4:
            r3 = True; break
    # Rule 4: would need 8 consecutive but we only have 7 - skip
    we_violations.append({
        "department": dept,
        "rule_1_3sigma":  r1,
        "rule_2_2of3":    r2,
        "rule_3_4of5":    r3,
        "any_violation":  bool(r1 or r2 or r3),
    })

we_df = pd.DataFrame(we_violations)
print("Western Electric pattern violations (within-department time series):")
print(we_df.to_string(index=False))

violators = we_df[we_df["any_violation"]]["department"].tolist()
print(f"\nDepartments with WE violations: {violators}")

we_df.to_csv(OUTPUTS_DIR / "spc_western_electric.csv", index=False)


## Step 8 - Cross-check: overall reorder rates by day-of-week and hour

A diagnostic to confirm there's a real day-of-week / hour signal in the data.

In [ ]:
dow_hour = pd.read_sql_query('''
    SELECT o.order_dow, o.order_hour_of_day,
           AVG(oi.reordered) AS reorder_rate, COUNT(*) AS n_items
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_dow, o.order_hour_of_day
''', conn)

heatmap = dow_hour.pivot(index="order_dow", columns="order_hour_of_day",
                          values="reorder_rate")
fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap, cmap="RdYlGn", center=heatmap.values.mean(),
            annot=False, cbar_kws={"label": "Reorder rate"}, ax=ax)
ax.set_title("Reorder Rate by Day-of-Week x Hour-of-Day", fontweight="bold")
ax.set_xlabel("Hour of day (0-23)")
ax.set_ylabel("Day of week (0=Sun .. 6=Sat)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dow_hour_heatmap.png", dpi=160, bbox_inches="tight")
plt.show()

conn.close()


**Notebook 03 complete.** Out-of-control departments identified and saved to `outputs/spc_department_stats.csv`. Move to `04_hypothesis_tests.ipynb`.